In [1]:
import math
import string
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt_tab')
nltk.download('stopwords')

file="/content/drive/MyDrive/IR_NLP_2025_Shared/Data/MELD DataSet/train_sent_emo.csv"

df=pd.read_csv(file)
df.head()

utterance=df['Utterance'].tolist()
sentiment=df['Sentiment'].tolist()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
def tokenize(sentences):
  tokenized_list=[]
  for i in sentences:
    i=i.replace('\x92',"'")
    i=i.replace('\x97','-')

    token=nltk.word_tokenize(i)
    token=[word.lower() for word in token if word not in string.punctuation]
    tokenized_list.append(token)

  return tokenized_list

def vocabulary(sentences):
  vocabulary=[]
  for i in sentences:
    for j in i:
      if j not in vocabulary:
        vocabulary.append(j)
  return vocabulary

tokenized_utterance=tokenize(utterance)

categories={
    'neutral':[],
    'negative':[],
    'positive':[]
}

for i in range(len(sentiment)):
  for j in range(len(tokenized_utterance[i])):
    categories[sentiment[i]].append(tokenized_utterance[i][j])

V=vocabulary(tokenized_utterance)

TF-IDF(t,d,D)=TF(t,d)×IDF(t,D)



In [3]:
from operator import inv
def inverted_index(V,classes):
  inv_idx={}

  for key in classes.keys():
    for word in V:
      if word in classes[key]:
        if word not in inv_idx.keys():
          inv_idx[word]=[(key,classes[key].count(word))]
        else:
          inv_idx[word].append((key,classes[key].count(word)))

  return inv_idx

inv_idx=inverted_index(V,categories)

print(inv_idx)

def tfidf_score(i_d,N):
  tfidf_dict={}
  for key in i_d.keys():
    tfidf_dict[key]=[]
    idf=math.log2(N/len(i_d[key]))
    for i,j in i_d[key]:
      tf=1+math.log2(1+j)
      tfidf_dict[key].append((i,tf*idf))

  return tfidf_dict

tfidf=tfidf_score(inv_idx,len(categories))

print(tfidf)

{'also': [('neutral', 10), ('negative', 3), ('positive', 8)], 'i': [('neutral', 1822), ('negative', 1431), ('positive', 869)], 'was': [('neutral', 212), ('negative', 178), ('positive', 122)], 'the': [('neutral', 917), ('negative', 596), ('positive', 339)], 'point': [('neutral', 7), ('negative', 2)], 'person': [('neutral', 8), ('negative', 9), ('positive', 4)], 'on': [('neutral', 225), ('negative', 172), ('positive', 101)], 'my': [('neutral', 204), ('negative', 294), ('positive', 167)], 'company': [('neutral', 2), ('negative', 2), ('positive', 1)], "'s": [('neutral', 844), ('negative', 514), ('positive', 439)], 'transition': [('neutral', 1)], 'from': [('neutral', 63), ('negative', 26), ('positive', 19)], 'kl-5': [('neutral', 1)], 'to': [('neutral', 783), ('negative', 549), ('positive', 295)], 'gr-6': [('neutral', 1)], 'system': [('neutral', 4), ('negative', 2), ('positive', 1)], 'you': [('neutral', 1558), ('negative', 1150), ('positive', 794)], 'must': [('neutral', 13), ('negative', 4),

In [4]:
# Organizza i valori per categoria
#from collections import defaultdict

top_words = {
    'neutral': [],
    'positive': [],
    'negative': []
}

for word,scores in tfidf.items():
    for category, value in scores:
        top_words[category].append((word, value))

top_10_per_category = {}
for category in top_words:
    sorted_words = sorted(top_words[category], key=lambda x: x[1], reverse=True)
    top_10_per_category[category] = sorted_words[:10]

# Stampa i risultati
for category, top_words in top_10_per_category.items():
    print(f"\nTop 10 parole per '{category}':")
    for word, value in top_words:
        print(f"{word}: {value}")


Top 10 parole per 'neutral':
2: 6.609174758105677
la: 6.609174758105677
30: 6.339850002884624
decide: 6.339850002884624
nope: 6.034514778397423
tall: 6.034514778397423
boutros: 6.034514778397423
keys: 6.034514778397423
breathe: 5.682031130134573
assistant: 5.682031130134573

Top 10 parole per 'positive':
yay: 6.850093961209695
incredible: 6.034514778397423
soo: 5.682031130134573
click: 5.265131460488539
notch: 4.754887502163468
ah-ha: 4.754887502163468
goodacre: 4.754887502163468
film: 4.754887502163468
chick-chick: 4.754887502163468
awesome: 4.754887502163468

Top 10 parole per 'negative':
ew: 7.266993630855729
horny: 6.034514778397423
pig: 6.034514778397423
vase: 5.265131460488539
during: 5.265131460488539
joke: 5.265131460488539
ace: 5.265131460488539
dammit: 5.265131460488539
horrible: 5.265131460488539
freaked: 5.265131460488539


In [5]:
#fai distribuzione di probabilità usando "categories", dividi in training e test set. usa NB classifier

{'neutral': ['also', 'i', 'was', 'the', 'point', 'person', 'on', 'my', 'company', "'s", 'transition', 'from', 'the', 'kl-5', 'to', 'gr-6', 'system', 'you', 'must', "'ve", 'had', 'your', 'hands', 'full', 'that', 'i', 'did', 'that', 'i', 'did', 'so', 'let', "'s", 'talk', 'a', 'little', 'bit', 'about', 'your', 'duties', 'now', 'you', "'ll", 'be', 'heading', 'a', 'whole', 'division', 'so', 'you', "'ll", 'have', 'a', 'lot', 'of', 'duties', 'i', 'see', 'but', 'there', "'ll", 'be', 'perhaps', '30', 'people', 'under', 'you', 'so', 'you', 'can', 'dump', 'a', 'certain', 'amount', 'on', 'them', 'good', 'to', 'know', 'we', 'can', 'go', 'into', 'detail', 'all', 'right', 'then', 'we', "'ll", 'have', 'a', 'definite', 'answer', 'for', 'you', 'on', 'monday', 'but', 'i', 'think', 'i', 'can', 'say', 'with', 'some', 'confidence', 'you', "'ll", 'fit', 'in', 'well', 'here', 'absolutely', 'you', 'can', 'relax', 'ok', 'all', 'right', 'well', '...', 'yeah', 'sure', 'hey', 'mon', 'hey-hey-hey', 'you', 'wan', 'n